# Load all datapack tables

This loader reads every `.csv`, `.csv.gz`, and `.jsonl.gz` file under `datapack/`. Monthly files for the same table are combined. Access data as `data[folder][table]`.

In [1]:
import sys
import pathlib
import time
import json
import gzip
import numpy as np
import pandas as pd
import os
from pathlib import Path

from IPython.display import display

In [2]:
DATA_ROOT = Path('datapack')

In [3]:
def _read_table_file(path):
    suffixes = path.suffixes
    if suffixes[-2:] == ['.jsonl', '.gz']:
        return pd.read_json(path, compression='gzip', lines=True)
    try:
        return pd.read_csv(path, compression='infer', encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, compression='infer', encoding='cp1252')
def _table_name(folder, path):
    stem = path.name.removesuffix('.gz').removesuffix('.csv').removesuffix('.jsonl')
    if folder in {'bookings', 'results', 'collections', 'invoices'}:
        return path.relative_to(DATA_ROOT / folder).parts[0] if folder == 'bookings' else folder
    if folder == 'payments':
        return 'gateway_export' if stem.startswith('gateway_export_') else 'settlements'
    if folder == 'campaigns':
        return 'campaign_bookings' if stem.startswith('campaign_bookings') else 'campaigns'
    return stem
def load_datapack(root=DATA_ROOT):
    """Load all CSV/CSV.GZ/JSONL.GZ files, grouped by folder and table."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Datapack folder not found: {root.resolve()}')
    supported = ('*.csv', '*.csv.gz', '*.jsonl.gz')
    paths = sorted(path for pattern in supported for path in root.rglob(pattern))
    grouped_frames = {}
    for path in paths:
        relative_path = path.relative_to(root)
        folder = relative_path.parts[0]
        table = _table_name(folder, path)
        try:
            frame = _read_table_file(path)
        except Exception as exc:
            raise RuntimeError(f'Could not load {relative_path}: {exc}') from exc
        frame['source_file'] = path.name
        frame['source_path'] = relative_path.as_posix()
        grouped_frames.setdefault((folder, table), []).append(frame)
    if not grouped_frames:
        raise FileNotFoundError(f'No supported data files found under {root.resolve()}')
    data = {}
    for (folder, table), frames in grouped_frames.items():
        data.setdefault(folder, {})[table] = pd.concat(frames, ignore_index=True, sort=False)
    return data
data = load_datapack()
# Convenience helpers for inspecting one folder at a time.
bookings_data = data['bookings']
def load_folder(folder):
    return data[folder]
print('Loaded tables:')
for folder, tables in data.items():
    for table, frame in tables.items():
        print(f'{folder}/{table}: {len(frame):,} rows, {len(frame.columns)} columns')


Loaded tables:
bookings/legacy: 1,252,786 rows, 14 columns
bookings/nova: 26,543 rows, 14 columns
campaigns/campaign_bookings: 48,869 rows, 5 columns
campaigns/campaigns: 1 rows, 7 columns
collections/collections: 552,492 rows, 8 columns
invoices/invoices: 470,776 rows, 9 columns
patients/patients: 309,217 rows, 8 columns
payments/gateway_export: 473,507 rows, 9 columns
payments/settlements: 562 rows, 9 columns
plan/plan_2026_H1: 6 rows, 10 columns
reference/cities: 6 rows, 5 columns
reference/clinics: 31 rows, 8 columns
reference/tests_catalogue: 24 rows, 6 columns
results/results: 1,078,420 rows, 7 columns


In [4]:
l = data['bookings']['legacy']

In [5]:
l.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1252786 entries, 0 to 1252785
Data columns (total 14 columns):
 #   Column        Non-Null Count    Dtype 
---  ------        --------------    ----- 
 0   booking_id    1252786 non-null  object
 1   created_at    1252786 non-null  object
 2   slot_at       1252786 non-null  object
 3   city          1252786 non-null  object
 4   clinic_id     1252786 non-null  object
 5   channel       1252786 non-null  object
 6   source        1252786 non-null  object
 7   patient_id    1252786 non-null  object
 8   patient_name  1252786 non-null  object
 9   test_code     1251496 non-null  object
 10  test_price    1252786 non-null  object
 11  home_fee      1252786 non-null  object
 12  source_file   1252786 non-null  object
 13  source_path   1252786 non-null  object
dtypes: object(14)
memory usage: 133.8+ MB


In [6]:
legacy = data['bookings']['legacy'].copy()
nova = data['bookings']['nova'].copy()
n_raw_legacy, n_raw_nova = len(legacy), len(nova)


---
# 1. Cleaning (per source system)

## 1.1 Legacy

**Latin-1 / non-UTF-8 fallback** — ✅ done in the loader (`_read_table_file`): UTF-8 first, then cp1252.

**Remove repeated header rows** — ✅ done, but it does not count them (it removed 21).

In [7]:
# : count the repeated header rows removed above (before - after) and print/log the number
n_before = len(legacy)
legacy = legacy[legacy["city"] != "city"].copy()
n_header_rows = n_before - len(legacy)
print(f"Removed {n_header_rows:,} repeated header rows")


Removed 21 repeated header rows


In [8]:
len(legacy)

1252765

**Parse `dd/mm/yyyy HH:MM` with an explicit format** — ⚠️ partly done. Currently `format="mixed"` + `dayfirst`, not a fixed format; bad dates become NaT but only `created_at` is checked.

In [9]:
# : parse legacy created_at / slot_at with an explicit format ("%d/%m/%Y %H:%M") instead of format="mixed"
# : count NaT in BOTH created_at and slot_at (not only created_at) and report them as rejects


In [10]:
for col in ["created_at", "slot_at"]:
    legacy[col] = pd.to_datetime(legacy[col], format="%d/%m/%Y %H:%M", errors="coerce")

n_bad_dates = legacy[["created_at", "slot_at"]].isna().sum()
print(n_bad_dates)


created_at    0
slot_at       0
dtype: int64


**Strip whitespace from `booking_id`** — ❌ missing

In [11]:
# : legacy booking_id -> .str.strip()
legacy["booking_id"] = legacy["booking_id"].str.strip()


**Dtypes** — ❌ missing (`test_price` and `home_fee` are still `object`)

In [12]:
# : legacy dtypes
#   city, clinic_id, channel -> category
#   test_price, home_fee   -> Int32 (pd.to_numeric(..., errors="coerce").astype("Int32"))
for col in ["city", "clinic_id", "channel"]:
    legacy[col] = legacy[col].astype("category")
for col in ["test_price", "home_fee"]:
    legacy[col] = pd.to_numeric(legacy[col], errors="coerce").astype("Int32")

print(legacy[["booking_id", "created_at", "slot_at", "city", "clinic_id", "channel", "test_price", "home_fee"]].dtypes)


booking_id            object
created_at    datetime64[ns]
slot_at       datetime64[ns]
city                category
clinic_id           category
channel             category
test_price             Int32
home_fee               Int32
dtype: object


## 1.2 Nova

**Flatten tests to one row per booking-test** — ✅ done

In [13]:
# Make nova's nested patient and test fields into columns and rows
nova["patient_id"] = nova["patient"].map(
    lambda x: x.get("patient_id") if isinstance(x, dict) else pd.NA
)
nova = nova.explode("tests", ignore_index=True)
nova["test_code"] = nova["tests"].map(
    lambda x: x.get("code") if isinstance(x, dict) else pd.NA
)
nova["test_price"] = nova["tests"].map(
    lambda x: x.get("price") if isinstance(x, dict) else pd.NA
)
print(f"Nova: {n_raw_nova:,} bookings -> {len(nova):,} booking-test rows")


Nova: 26,543 bookings -> 62,439 booking-test rows


**Lowercase `channel` and `source`** — ❌ missing

In [14]:
# : nova channel, source -> .str.strip().str.lower()
for col in ["channel", "source"]:
    nova[col] = nova[col].str.strip().str.lower()



**UTC -> IST, drop the timezone** — ❌ done wrongly. Everything is currently converted to UTC and legacy is not handled. Nova timestamps are UTC (convert to Asia/Kolkata, then drop tz); legacy timestamps are already IST local time (only make them tz-naive).

In [15]:
for col in ["created_at", "slot_at"]:
    nova[col] = (
        pd.to_datetime(nova[col], utc=True)
        .dt.tz_convert("Asia/Kolkata")
        .dt.tz_localize(None)
    )
print(nova[["created_at", "slot_at"]].dtypes)
print(nova[["created_at", "slot_at"]].head(3))


created_at    datetime64[ns]
slot_at       datetime64[ns]
dtype: object
           created_at             slot_at
0 2026-04-21 01:57:00 2026-04-25 10:16:00
1 2026-03-28 03:02:00 2026-04-01 09:17:00
2 2026-03-28 03:02:00 2026-04-01 09:17:00


In [16]:
# bookings_all["created_at"] = bookings_all["created_at"].dt.tz_convert("Asia/Kolkata")

In [17]:
common_columns = [
    "booking_id", "created_at", "slot_at", "city", "clinic_id",
    "channel", "patient_id", "test_code", "test_price", "home_fee",
    "rescheduled_from", "status_at_export", "source_file", "source_path", "source"
]
legacy["booking_system"] = "legacy"
nova["booking_system"] = "nova"

bookings_all = pd.concat(
    [
        legacy.reindex(columns=common_columns + ["booking_system"]),
        nova.reindex(columns=common_columns + ["booking_system"]),
    ],
    ignore_index=True,
)

# Baseline for the reconciliation assert in phase F
n_raw_rows = len(bookings_all)
print(f"Raw booking-test rows: {n_raw_rows:,}")
print(bookings_all.dtypes)


Raw booking-test rows: 1,315,204
booking_id                  object
created_at          datetime64[ns]
slot_at             datetime64[ns]
city                        object
clinic_id                   object
channel                     object
patient_id                  object
test_code                   object
test_price                  object
home_fee                     Int64
rescheduled_from            object
status_at_export            object
source_file                 object
source_path                 object
source                      object
booking_system              object
dtype: object


In [18]:
for col in ["test_price", "home_fee"]:
    bookings_all[col] = pd.to_numeric(bookings_all[col], errors="coerce").astype("Int32")


In [19]:
bookings_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1315204 entries, 0 to 1315203
Data columns (total 16 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   booking_id        1315204 non-null  object        
 1   created_at        1315204 non-null  datetime64[ns]
 2   slot_at           1315204 non-null  datetime64[ns]
 3   city              1315204 non-null  object        
 4   clinic_id         1315204 non-null  object        
 5   channel           1315204 non-null  object        
 6   patient_id        1315204 non-null  object        
 7   test_code         1313893 non-null  object        
 8   test_price        1315183 non-null  Int32         
 9   home_fee          1315204 non-null  Int32         
 10  rescheduled_from  3542 non-null     object        
 11  status_at_export  62439 non-null    object        
 12  source_file       1315204 non-null  object        
 13  source_path       1315204 non-null  object

---
# 2. Validation (normalise or reject)
Nothing is silently fixed: every row is either normalised or written to `rejects_bookings.csv` with a `reason`.

## 2.1 Normalise

**Strip whitespace** — ❌ missing

In [20]:
# : strip leading/trailing whitespace on all string columns (booking_id, city, clinic_id, channel, source, test_code, patient_id)
str_cols = ["booking_id", "city", "clinic_id", "channel", "source", "test_code", "patient_id"]
for col in str_cols:
    bookings_all[col] = bookings_all[col].str.strip()


**Lowercase `channel` and `source`** — ❌ missing

In [21]:
# : channel, source -> .str.lower()
for col in ["channel", "source"]:
    bookings_all[col] = bookings_all[col].str.lower()


**Bangalore -> Bengaluru** — ❌ missing

In [22]:
# : city -> replace "Bangalore" with "Bengaluru"
n_bangalore = bookings_all["city"].eq("Bangalore").sum()
bookings_all["city"] = bookings_all["city"].replace({"Bangalore": "Bengaluru"})
print(f"Renamed {n_bangalore:,} Bangalore rows to Bengaluru")
print(sorted(bookings_all["city"].unique()))


Renamed 193,558 Bangalore rows to Bengaluru
['Ahmedabad', 'Bengaluru', 'Chennai', 'Hyderabad', 'Jaipur', 'Pune']


In [23]:
bookings_all["channel"].unique()

array(['clinic', 'home'], dtype=object)

In [24]:
bookings_all["source"].unique()

array(['walkin', 'app', 'phone'], dtype=object)

**Uppercase test codes** — ❌ missing

In [25]:
# : test_code -> .str.upper()
bookings_all["test_code"] = bookings_all["test_code"].str.upper()

In [26]:
bookings_all.shape

(1315204, 16)

## 2.2 Reject

In [27]:
# : set up a rejects collection, e.g.
#   rejects = []   # list of DataFrames, each with a `reason` column
#   def reject(df, mask, reason): rejects.append(df.loc[mask].assign(reason=reason)); return df.loc[~mask]
rejects = []

def reject(df, mask, reason):
    """Move rows where mask is True to `rejects` with a reason; return the rest."""
    mask = mask.fillna(True)  # a null in the checked field counts as a failure
    rejects.append(df.loc[mask].assign(reason=reason))
    return df.loc[~mask]


In [28]:
rejects

[]

**Booking ID doesn't match the legacy or Nova pattern** — ❌ missing

In [29]:
# : reject booking_id not matching the legacy regex or the Nova regex (check the data dictionary for both patterns)
LEGACY_ID = r"^BK-[A-Z]{3}-\d{6}-\d{6}$"
NOVA_ID = r"^nova_[a-z]{3}_\d{7}$"

valid_id = (
    (bookings_all["booking_system"].eq("legacy") & bookings_all["booking_id"].str.match(LEGACY_ID))
    | (bookings_all["booking_system"].eq("nova") & bookings_all["booking_id"].str.match(NOVA_ID))
)
bookings_all["id_flag"] = ~valid_id   # kept: collection app and lab carry the same id
print(f"Flagged {bookings_all['id_flag'].sum():,} rows, "
      f"{bookings_all.loc[bookings_all['id_flag'], 'booking_id'].nunique():,} bookings")


Flagged 1,839 rows, 793 bookings


**Test code isn't `T###` or `PK##`** — ❌ missing. The 1,290 null `test_code` rows would also be rejected here.

In [30]:
no_tests = bookings_all["test_code"].isna() & bookings_all["test_price"].isna()
bookings_all = reject(bookings_all, no_tests, "no tests")
print(f"no tests rejected: {len(rejects[-1]):,}")


no tests rejected: 21


In [31]:
bookings_all["created_after_slot"] = bookings_all["slot_at"] < bookings_all["created_at"]   # kept: slot_at is the one used
print(f"created_at after slot_at, kept: {bookings_all['created_after_slot'].sum():,}")


created_at after slot_at, kept: 2,443


In [32]:
bookings_all["test_code"].nunique()

24

**Negative price** — ❌ missing

In [33]:
# : reject test_price < 0
bookings_all = reject(bookings_all, bookings_all["test_price"].lt(0), "negative test_price")
print(f"negative test_price: {len(rejects[-1]):,}")


negative test_price: 0


**`slot_at` before `created_at`** — ❌ missing

In [34]:
# : reject slot_at < created_at
bookings_all = reject(bookings_all, bookings_all["slot_at"] < bookings_all["created_at"], "slot_at before created_at")
print(f"slot_at before created_at: {len(rejects[-1]):,}")


slot_at before created_at: 2,443


In [35]:
bad_set = ~bookings_all["channel"].isin(["clinic", "home"]) | ~bookings_all["source"].isin(["app", "phone", "walkin"])
bookings_all = reject(bookings_all, bad_set, "channel/source not allowed")
print(f"channel/source: {len(rejects[-1]):,}")
print(f"Remaining: {len(bookings_all):,}")


channel/source: 0
Remaining: 1,312,740


**Booking with no tests** — ⚠️ partly done. The 21 rows with no price are dropped below, but silently with no reason recorded. This breaks "nothing is silently fixed".

In [36]:
bookings_all[bookings_all["test_price"].isna()]

,booking_id,created_at,slot_at,city,clinic_id,channel,patient_id,test_code,test_price,home_fee,rescheduled_from,status_at_export,source_file,source_path,source,booking_system,id_flag,created_after_slot


In [37]:
bookings_all[bookings_all["test_price"].isna()].count()

booking_id            0
created_at            0
slot_at               0
city                  0
clinic_id             0
channel               0
patient_id            0
test_code             0
test_price            0
home_fee              0
rescheduled_from      0
status_at_export      0
source_file           0
source_path           0
source                0
booking_system        0
id_flag               0
created_after_slot    0
dtype: int64

In [38]:
bookings_all = bookings_all.loc[bookings_all["test_price"].notna()].copy()

print(bookings_all[["test_code", "test_price"]].isna().sum())
print(bookings_all["test_price"].dtype)

test_code     1289
test_price       0
dtype: int64
Int32


**Channel or source outside the allowed set** — ❌ missing

In [39]:
rejects_df = pd.concat(rejects, ignore_index=True)
rejects_df.to_csv("rejects_bookings.csv", index=False)

summary = (
    rejects_df.groupby("reason")
    .agg(rows=("booking_id", "size"), bookings=("booking_id", "nunique"))
)
print(summary)
print(f"Total: {len(rejects_df):,} rows, {rejects_df['booking_id'].nunique():,} bookings")


                           rows  bookings
reason                                   
no tests                     21        21
slot_at before created_at  2443      1030
Total: 2,464 rows, 1,051 bookings


In [40]:
# : reject channel / source not in the allowed sets (see data dictionary)
accepted = bookings_all
assert n_raw_rows == len(accepted) + len(rejects_df), (
    n_raw_rows, len(accepted), len(rejects_df)
)
print(f"raw {n_raw_rows:,} = accepted {len(accepted):,} + rejected {len(rejects_df):,}")


raw 1,315,204 = accepted 1,312,740 + rejected 2,464


## 2.3 Output and reconciliation

In [41]:
bookings_all["test_code"].value_counts()

test_code
T001    161437
T002    130201
T004     96418
T003     87576
T007     87288
T010     78150
T005     68930
PK01     66013
T006     59917
T008     59615
T009     49612
PK02     45258
PK03     45059
T013     40397
T014     40245
T011     30477
T015     30452
T016     30330
T017     30204
PK04     22720
T012     20323
T019     10344
T018     10269
T020     10216
Name: count, dtype: int64

In [42]:
bookings_all["has_package_row"] = bookings_all["test_code"].str.startswith("PK")

grp = bookings_all.groupby("booking_id", sort=False)
bookings = grp.agg(
    n_tests=("test_code", "count"),
    price_sum=("test_price", "sum"),
    home_fee=("home_fee", "first"),
    has_package=("has_package_row", "any"),
    created_at=("created_at", "first"),
    slot_at=("slot_at", "first"),
    city=("city", "first"),
    clinic_id=("clinic_id", "first"),
    channel=("channel", "first"),
    source=("source", "first"),
    patient_id=("patient_id", "first"),
    rescheduled_from=("rescheduled_from", "first"),
    status_at_export=("status_at_export", "first"),
    booking_system=("booking_system", "first"),
).reset_index()

bookings["amount_rs"] = (bookings["price_sum"] + bookings["home_fee"]).astype("int64")
bookings = bookings.drop(columns="price_sum")
bookings["month"] = bookings["slot_at"].dt.to_period("M").astype(str)
bookings["is_reschedule"] = bookings["rescheduled_from"].notna()

print(f"{len(bookings):,} bookings, {int(bookings['n_tests'].sum()):,} tests")


566,956 bookings, 1,311,451 tests


In [43]:
bookings

,booking_id,n_tests,home_fee,has_package,created_at,slot_at,city,clinic_id,channel,source,patient_id,rescheduled_from,status_at_export,booking_system,amount_rs,month,is_reschedule
0,BK-AHM-202501-000001,3,0,False,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,None,None,legacy,2000,2025-01,False
1,BK-AHM-202501-000002,1,0,False,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,None,None,legacy,650,2025-01,False
2,BK-AHM-202501-000003,4,150,False,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,None,None,legacy,1820,2025-01,False
3,BK-AHM-202501-000004,1,0,False,2025-01-20 02:48:00,2025-01-24 07:15:00,Ahmedabad,AHM-03,clinic,app,PAHM0007069,None,None,legacy,160,2025-01,False
4,BK-AHM-202501-000005,2,0,False,2025-01-06 08:21:00,2025-01-11 09:56:00,Ahmedabad,AHM-03,clinic,app,PAHM0002520,None,None,legacy,1900,2025-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566951,nova_jai_0055749,2,150,False,2026-06-02 04:49:00,2026-06-08 11:54:00,Jaipur,JAI-02,home,app,PJAI0028824,nova_jai_0054922,booked,nova,1150,2026-06,True
566952,nova_jai_0055750,3,150,False,2026-06-15 23:28:00,2026-06-18 08:22:00,Jaipur,JAI-01,home,walkin,PJAI0028764,nova_jai_0054877,booked,nova,1950,2026-06,True
566953,nova_jai_0055751,1,0,True,2026-06-19 05:36:00,2026-06-26 11:47:00,Jaipur,JAI-02,clinic,walkin,PJAI0028485,nova_jai_0052936,booked,nova,1499,2026-06,True
566954,nova_jai_0055752,3,150,False,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,nova_jai_0052790,booked,nova,1470,2026-06,True


In [44]:
is_superseded = bookings["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
current_bookings = bookings.loc[~is_superseded].copy()

print(f"Removed {is_superseded.sum():,} superseded Nova bookings")
print(f"Kept {len(current_bookings):,} bookings, {int(current_bookings['n_tests'].sum()):,} tests")
print(f"Reschedule replacements: {current_bookings['is_reschedule'].sum():,}")


Removed 1,499 superseded Nova bookings
Kept 565,457 bookings, 1,307,910 tests
Reschedule replacements: 1,500


In [45]:
bookings['patient_id'].count()

np.int64(566956)

## Loading Collections

In [46]:
coll = data["collections"]["collections"].copy()

In [47]:
coll["status"].value_counts()

status
collected    473320
no_show       79172
Name: count, dtype: int64

In [48]:
coll["booking_id"] = coll["booking_id"].str.strip()

In [49]:
coll["collected_at"] = pd.to_datetime(coll["collected_at"], errors="coerce")

In [50]:
coll[["booking_id", "status", "collected_at"]].value_counts()

booking_id            status     collected_at       
BK-BLR-202506-039771  collected  2025-06-22 06:27:00    2
BK-AHM-202501-003275  collected  2025-01-08 07:47:00    2
BK-BLR-202506-042267  collected  2025-06-20 11:25:00    2
BK-CHE-202502-010328  collected  2025-02-19 18:38:00    2
BK-BLR-202604-109137  collected  2026-04-10 08:55:00    2
                                                       ..
nova_jai_0055577      collected  2026-06-01 17:31:00    1
nova_jai_0055578      collected  2026-06-27 11:41:00    1
nova_jai_0055579      collected  2026-06-06 06:29:00    1
nova_jai_0055580      collected  2026-06-10 13:14:00    1
BK-AHM-202501-000031  collected  2025-01-19 16:25:00    1
Name: count, Length: 470426, dtype: int64

In [51]:
coll[coll["booking_id"]=="BK-AHM-202501-000032"]

,booking_id,clinic_id,channel,status,collected_at,phlebotomist_id,source_file,source_path
30,BK-AHM-202501-000032,AHM-04,clinic,collected,2025-01-02 12:22:00,NaN,collections_AHM_2025-01.csv.gz,collections/collections_AHM_2025-01.csv.gz
5446,BK-AHM-202501-000032,AHM-04,clinic,collected,2025-01-02 12:22:00,NaN,collections_AHM_2025-01.csv.gz,collections/collections_AHM_2025-01.csv.gz


In [52]:
n_coll_in = len(coll)
dups = coll.duplicated(subset=["booking_id", "status", "collected_at"], keep="first")

In [53]:
coll_clean = coll.loc[~dups].copy()

In [54]:


print(f"Collections: {n_coll_in:,} in, {dups.sum():,} double-scans removed, {len(coll_clean):,} kept")
assert n_coll_in == len(coll_clean) + dups.sum()
assert coll_clean["booking_id"].is_unique

Collections: 552,492 in, 2,894 double-scans removed, 549,598 kept


**Left-join collections** (`one_to_one`, no row fan-out) — ❌ missing

In [55]:
n_before = len(current_bookings)
joined = current_bookings.merge(
    coll_clean[["booking_id", "status", "collected_at", "phlebotomist_id"]],
    on="booking_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)
assert len(joined) == n_before, "join fanned out"
print(joined["_merge"].value_counts())


_merge
both          547139
left_only      18318
right_only         0
Name: count, dtype: int64


In [56]:
joined["status"] = joined["status"].fillna("cancelled_or_missing")
print(joined["status"].value_counts())


status
collected               468320
no_show                  78819
cancelled_or_missing     18318
Name: count, dtype: int64


In [57]:
joined

,booking_id,n_tests,home_fee,has_package,created_at,slot_at,city,clinic_id,channel,source,...,rescheduled_from,status_at_export,booking_system,amount_rs,month,is_reschedule,status,collected_at,phlebotomist_id,_merge
0,BK-AHM-202501-000001,3,0,False,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,...,None,None,legacy,2000,2025-01,False,collected,2025-01-09 09:09:00,NaN,both
1,BK-AHM-202501-000002,1,0,False,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,...,None,None,legacy,650,2025-01,False,collected,2025-01-06 13:44:00,NaN,both
2,BK-AHM-202501-000003,4,150,False,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,...,None,None,legacy,1820,2025-01,False,collected,2025-01-28 13:37:00,PHAHM047,both
3,BK-AHM-202501-000004,1,0,False,2025-01-20 02:48:00,2025-01-24 07:15:00,Ahmedabad,AHM-03,clinic,app,...,None,None,legacy,160,2025-01,False,cancelled_or_missing,NaT,NaN,left_only
4,BK-AHM-202501-000005,2,0,False,2025-01-06 08:21:00,2025-01-11 09:56:00,Ahmedabad,AHM-03,clinic,app,...,None,None,legacy,1900,2025-01,False,collected,2025-01-11 09:59:00,NaN,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565452,nova_jai_0055749,2,150,False,2026-06-02 04:49:00,2026-06-08 11:54:00,Jaipur,JAI-02,home,app,...,nova_jai_0054922,booked,nova,1150,2026-06,True,cancelled_or_missing,NaT,NaN,left_only
565453,nova_jai_0055750,3,150,False,2026-06-15 23:28:00,2026-06-18 08:22:00,Jaipur,JAI-01,home,walkin,...,nova_jai_0054877,booked,nova,1950,2026-06,True,cancelled_or_missing,NaT,NaN,left_only
565454,nova_jai_0055751,1,0,True,2026-06-19 05:36:00,2026-06-26 11:47:00,Jaipur,JAI-02,clinic,walkin,...,nova_jai_0052936,booked,nova,1499,2026-06,True,cancelled_or_missing,NaT,NaN,left_only
565455,nova_jai_0055752,3,150,False,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,...,nova_jai_0052790,booked,nova,1470,2026-06,True,cancelled_or_missing,NaT,NaN,left_only


In [58]:
skew = (joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)).fillna(False)

In [59]:
skew

0         False
1         False
2         False
3         False
4         False
          ...  
565452    False
565453    False
565454    False
565455    False
565456    False
Length: 565457, dtype: bool

### **

In [60]:
skew = (joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)).fillna(False)
joined["flag_clock_skew"] = skew
joined.loc[skew, "collected_at"] = pd.NaT
bookings_full = joined.drop(columns="_merge")
print(f"Clock skew: {int(skew.sum()):,}")
print(f"collected_at null: {bookings_full['collected_at'].isna().sum():,}")


Clock skew: 939
collected_at null: 98,076


In [61]:
pat = data["patients"]["patients"].copy()

pat["phone_norm"] = (
    pat["phone"].astype("string").str.replace(r"\D", "", regex=True).str[-10:]
)
pat["name_norm"] = (
    pat["patient_name"].astype("string")
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

keys = ["phone_norm", "name_norm", "dob"]

# Only group patients with all three identifying details.
valid = pat[keys].notna().all(axis=1)

# Choose the first patient_id in each group as that person's ID.
first_per_person = (
    pat.loc[valid, keys + ["patient_id"]]
    .drop_duplicates(subset=keys, keep="first")
    .rename(columns={"patient_id": "person_id"})
)

id_map = pat.loc[valid, ["patient_id"] + keys].merge(
    first_per_person,
    on=keys,
    how="left",
    validate="many_to_one",
)[["patient_id", "person_id"]]

# If any identifying detail is missing, keep that row's own ID.
unmatched = pat.loc[~valid, ["patient_id"]].rename(
    columns={"patient_id": "person_id"}
)
unmatched["patient_id"] = pat.loc[~valid, "patient_id"].to_numpy()

id_map = pd.concat([id_map, unmatched], ignore_index=True)

bookings_final = bookings_full.merge(
    id_map,
    on="patient_id",
    how="left",
    validate="many_to_one",
)

In [62]:
assert len(bookings_final) == len(bookings_full), "join fanned out"
assert bookings_final["person_id"].notna().all(), "unmapped patient_id"
print(f"{len(bookings_final):,} bookings, {bookings_final['person_id'].nunique():,} distinct persons")
print(f"patient_id changed to person_id on {(bookings_final['patient_id'] != bookings_final['person_id']).sum():,} bookings")

565,457 bookings, 188,210 distinct persons
patient_id changed to person_id on 5,862 bookings


In [63]:
bookings_2025_h1 = bookings_final[bookings_final["slot_at"].between("2025-01-01", "2025-07-01", inclusive="left")]
bookings_2026_h1 = bookings_final[bookings_final["slot_at"].between("2026-01-01", "2026-07-01", inclusive="left")]


In [64]:
print(f"{len(bookings_2025_h1):,} bookings in 2025 H1, {len(bookings_2026_h1):,} in 2026 H1")


189,500 bookings in 2025 H1, 194,713 in 2026 H1


In [65]:
bookings_final.to_csv("bookings_final.csv", index=False)

In [66]:
t25 = bookings_2025_h1["n_tests"].sum()
t26 = bookings_2026_h1["n_tests"].sum()
print(f"Tests:     {t25:,} -> {t26:,}  ({t26 / t25 - 1:+.2%})")
print(f"Bookings:  {len(bookings_2025_h1):,} -> {len(bookings_2026_h1):,}  ({len(bookings_2026_h1) / len(bookings_2025_h1) - 1:+.2%})")
p25, p26 = bookings_2025_h1["person_id"].nunique(), bookings_2026_h1["person_id"].nunique()
print(f"Patients:  {p25:,} -> {p26:,}  ({p26 / p25 - 1:+.2%})")


Tests:     435,008 -> 456,788  (+5.01%)
Bookings:  189,500 -> 194,713  (+2.75%)
Patients:  94,477 -> 98,025  (+3.76%)


In [67]:
%pip install Jinja2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
by_city = pd.concat(
    [
        bookings_2025_h1.groupby("city")["n_tests"].sum().rename("tests_2025"),
        bookings_2026_h1.groupby("city")["n_tests"].sum().rename("tests_2026"),
    ],
    axis=1,
)
by_city["growth"] = by_city["tests_2026"] / by_city["tests_2025"] - 1
by_city.loc["Total"] = [by_city["tests_2025"].sum(), by_city["tests_2026"].sum(), t26 / t25 - 1]
by_city.style.format({"tests_2025": "{:,.0f}", "tests_2026": "{:,.0f}", "growth": "{:+.1%}"})


,tests_2025,tests_2026,growth
city,,,
Ahmedabad,"76,727","84,816",+10.5%
Bengaluru,"99,732","107,355",+7.6%
Chennai,"80,022","80,636",+0.8%
Hyderabad,"73,058","78,450",+7.4%
Jaipur,"45,017","40,164",-10.8%
Pune,"60,452","65,367",+8.1%
Total,"435,008","456,788",+5.0%


In [69]:
def tree(d):
    patients = d["person_id"].nunique()
    visits = len(d)
    tests = d["n_tests"].sum()
    return pd.Series({
        "patients": patients,
        "visits_per_patient": visits / patients,
        "tests_per_visit": tests / visits,
        "tests": tests,
    })

def decompose(d25, d26):
    a, b = tree(d25), tree(d26)
    out = pd.DataFrame({"2025": a, "2026": b})
    out["growth"] = round((out["2026"] / out["2025"] - 1) * 100, 2)
    return out

overall = decompose(bookings_2025_h1, bookings_2026_h1)
overall


,2025,2026,growth
patients,94477.000000,98025.000000,3.76
visits_per_patient,2.005779,1.986361,-0.97
tests_per_visit,2.295557,2.345955,2.20
tests,435008.000000,456788.000000,5.01


In [70]:
g = overall["growth"] / 100

parts = (1 + g["patients"]) * (1 + g["visits_per_patient"]) * (1 + g["tests_per_visit"])
target = 1 + g["tests"]

print("Component product:", parts)
print("Total tests factor:", target)
print("Difference:", parts - target)
print("\nGrowth inputs:")
print(g)
assert np.isclose(parts, target, atol=1e-4)


Component product: 1.05014105616
Total tests factor: 1.0501
Difference: 4.10561599999415e-05

Growth inputs:
patients              0.0376
visits_per_patient   -0.0097
tests_per_visit       0.0220
tests                 0.0501
Name: growth, dtype: float64


In [71]:
rows = {}
for city in sorted(bookings_final["city"].unique()):
    t = decompose(
        bookings_2025_h1[bookings_2025_h1["city"] == city],
        bookings_2026_h1[bookings_2026_h1["city"] == city],
    )["growth"]
    rows[city] = t
by_city_tree = pd.DataFrame(rows).T
by_city_tree.style.format("{:+.1f}%")


,patients,visits_per_patient,tests_per_visit,tests
Ahmedabad,+9.6%,-1.0%,+1.9%,+10.5%
Bengaluru,+7.2%,-2.3%,+2.8%,+7.6%
Chennai,-5.0%,+3.2%,+2.8%,+0.8%
Hyderabad,+8.1%,-2.5%,+1.9%,+7.4%
Jaipur,-12.1%,-0.3%,+1.8%,-10.8%
Pune,+8.9%,-2.3%,+1.6%,+8.1%


In [72]:
mig = bookings_final[
    bookings_final["city"].isin(["Chennai", "Jaipur"])
    & (bookings_final["slot_at"].dt.month <= 6)
].copy()
mig["year"] = mig["slot_at"].dt.year
mig["quarter"] = np.where(mig["slot_at"].dt.month <= 3, "Q1", "Q2")

q = mig.groupby(["city", "quarter", "year"]).agg(
    visits=("booking_id", "size"),
    patients=("person_id", "nunique"),
    tests=("n_tests", "sum"),
).unstack("year")
q["visits_growth"] = (q[("visits", 2026)] / q[("visits", 2025)] - 1)*100
q["patients_growth"] = (q[("patients", 2026)] / q[("patients", 2025)] - 1)*100
q


visits        patients         tests        visits_growth  \
year              2025   2026     2025   2026   2025   2026                 
city    quarter                                                             
Chennai Q1       17429  17609    11084  10941  40090  41610      1.032761   
        Q2       17458  16585    11213  10466  39932  39026     -5.000573   
Jaipur  Q1       10427   8775     6427   5531  23943  20420    -15.843483   
        Q2        9161   8394     5934   5388  21074  19744     -8.372448   

                patients_growth  
year                             
city    quarter                  
Chennai Q1            -1.290148  
        Q2            -6.661910  
Jaipur  Q1           -13.941186  
        Q2            -9.201213

In [73]:
res = data["results"]["results"].copy()
res["booking_id"] = res["booking_id"].str.strip()
result_tests = res.groupby("booking_id").size().rename("result_tests")

x = bookings_final.merge(result_tests, on="booking_id", how="left")
x["result_tests"] = x["result_tests"].fillna(0)
x["collected_tests"] = np.where(x["status"].eq("collected"), x["n_tests"], 0)

x = x[
    x["city"].isin(["Chennai", "Jaipur"])
    & (x["slot_at"].dt.month <= 6)
    & x["slot_at"].dt.year.isin([2025, 2026])
].copy()
x["year"] = x["slot_at"].dt.year
x["quarter"] = np.where(x["slot_at"].dt.month <= 3, "Q1", "Q2")

src = (
    x.groupby(["city", "quarter", "year"])[["n_tests", "collected_tests", "result_tests"]]
    .sum()
    .unstack("year")
)
for col in ["n_tests", "collected_tests", "result_tests"]:
    src[(col, "growth")] = src[(col, 2026)] / src[(col, 2025)] - 1

growth = src.xs("growth", axis=1, level=1)
growth.style.format("{:+.1%}")


In [74]:
y = x.copy()
y["system"] = np.where(y["booking_id"].str.startswith("nova_"), "nova", "legacy")

diag = y.groupby(["city", "quarter", "year"]).agg(
    booked=("n_tests", "sum"),
    collected=("collected_tests", "sum"),
    results=("result_tests", "sum"),
    bookings=("booking_id", "size"),
    no_show=("status", lambda s: s.eq("no_show").sum()),
    cancelled=("status", lambda s: s.eq("cancelled_or_missing").sum()),
)
diag["results_per_booked_test"] = diag["results"] / diag["booked"]
diag["no_show_rate"] = diag["no_show"] / diag["bookings"]
diag["cancelled_rate"] = diag["cancelled"] / diag["bookings"]
diag[["results_per_booked_test", "no_show_rate", "cancelled_rate"]].style.format("{:.1%}")


In [76]:
res_tests = data["results"]["results"].assign(
    booking_id=lambda d: d["booking_id"].str.strip()
).groupby("booking_id").size().rename("result_tests")

z = bookings_final.merge(res_tests, on="booking_id", how="left")
z["result_tests"] = z["result_tests"].fillna(0)
z["collected_tests"] = np.where(z["status"].eq("collected"), z["n_tests"], 0)

def h1(d, year):
    return d[(d["slot_at"] >= f"{year}-01-01") & (d["slot_at"] < f"{year}-07-01")]

cols = ["n_tests", "collected_tests", "result_tests"]
a = h1(z, 2025).groupby("city")[cols].sum()
b = h1(z, 2026).groupby("city")[cols].sum()
g = b / a - 1
g.loc["Total"] = b.sum() / a.sum() - 1

plan = data["plan"]["plan_2026_H1"].copy()
plan["city"] = plan["city"].replace({"Bangalore": "Bengaluru"})
plan = plan.set_index("city")
month_cols = [c for c in plan.columns if c.startswith("2026-")]
plan_h1 = plan[month_cols].sum(axis=1)

out = g.copy()
out["plan_2026"] = plan_h1
out.loc["Total", "plan_2026"] = plan_h1.sum()



out["booked_vs_plan"] = b["n_tests"].reindex(out.index).fillna(b["n_tests"].sum()) / out["plan_2026"] - 1
out["results_vs_plan"] = b["result_tests"].reindex(out.index).fillna(b["result_tests"].sum()) / out["plan_2026"] - 1
out.style.format({c: "{:+.1%}" for c in cols + ["booked_vs_plan", "results_vs_plan"]} | {"plan_2026": "{:,.0f}"})


,n_tests,collected_tests,result_tests,plan_2026,booked_vs_plan,results_vs_plan
city,,,,,,
Ahmedabad,+10.5%,+10.8%,+11.0%,"90,773",-6.6%,-23.3%
Bengaluru,+7.6%,+9.0%,+9.0%,"120,085",-10.6%,-26.3%
Chennai,+0.8%,-1.4%,-1.4%,"93,912",-14.1%,-31.7%
Hyderabad,+7.4%,+7.8%,+7.9%,"86,421",-9.2%,-25.3%
Jaipur,-10.8%,-12.6%,-12.6%,"52,372",-23.3%,-38.5%
Pune,+8.1%,+8.7%,+8.7%,"72,177",-9.4%,-25.5%
Total,+5.0%,+4.9%,+5.0%,"515,740",-11.4%,-27.7%


---
# 3. Board deck numbers (Brief A, Brief B and extras)

Every figure in `Kalpa_Health_H1_2026_Growth_Review.pptx` comes from the cells below and is written to `deck_metrics.json`. The deck builder reads that file, so the deck and this notebook cannot drift apart.

| Tag | Meaning |
|---|---|
| **A** | Brief A: which part of the business is short |
| **B** | Brief B: why bookings fell in Chennai and Jaipur |
| **EXTRA** | Not asked for in A or B (channel mix, revenue). Kept, but labelled as extra |

`DECK-METRICS` marker: do not delete this cell, it stops the section being added twice.

In [ ]:
# DECK-METRICS: base frame. `z` = bookings_final + lab results + collected tests (built in section 2).
import json

dm = z.copy()
dm["year"] = dm["slot_at"].dt.year
dm["m"] = dm["slot_at"].dt.month
d25 = dm[(dm["year"] == 2025) & (dm["m"] <= 6)]
d26 = dm[(dm["year"] == 2026) & (dm["m"] <= 6)]

def dm_tree(d):
    p, v, t = d["person_id"].nunique(), len(d), d["n_tests"].sum()
    return dict(patients=int(p), visits=int(v), tests=int(t), vpp=v / p, tpv=t / v)

def dm_decompose(a_, b_):
    x_, y_ = dm_tree(a_), dm_tree(b_)
    return dict(y25=x_, y26=y_, g={k: y_[k] / x_[k] - 1 for k in x_})

deck = {}
# A: headline tree, overall and by city. Extra: by channel.
deck["overall"] = dm_decompose(d25, d26)
deck["city"] = {c: dm_decompose(d25[d25["city"] == c], d26[d26["city"] == c]) for c in sorted(dm["city"].unique())}
deck["channel"] = {c: dm_decompose(d25[d25["channel"] == c], d26[d26["channel"] == c]) for c in ["clinic", "home"]}
print(f"Tests {deck['overall']['y25']['tests']:,} -> {deck['overall']['y26']['tests']:,} ({deck['overall']['g']['tests']:+.2%})")


In [ ]:
# A: plan vs actual by city, shortfall, and the three-count cross-check (booked / collected / resulted)
plan_df = data["plan"]["plan_2026_H1"].copy()
plan_df["city"] = plan_df["city"].replace({"Bangalore": "Bengaluru"})
plan_df = plan_df.set_index("city")
plan_months = [c for c in plan_df.columns if c.startswith("2026-")]
plan_h1_city = plan_df[plan_months].sum(axis=1)

cities = {}
for c in plan_h1_city.index:
    a_, b_ = d25[d25["city"] == c], d26[d26["city"] == c]
    t25_, t26_ = int(a_["n_tests"].sum()), int(b_["n_tests"].sum())
    cities[c] = dict(
        t25=t25_, t26=t26_, plan=int(plan_h1_city[c]),
        growth=t26_ / t25_ - 1, plan_growth=plan_h1_city[c] / t25_ - 1, short=int(plan_h1_city[c] - t26_),
        collected25=int(a_["collected_tests"].sum()), collected26=int(b_["collected_tests"].sum()),
        res25=int(a_["result_tests"].sum()), res26=int(b_["result_tests"].sum()),
    )
deck["plan"] = dict(
    cities=cities, total_plan=int(plan_h1_city.sum()),
    total25=int(d25["n_tests"].sum()), total26=int(d26["n_tests"].sum()),
    by_month={k: int(plan_df[k].sum()) for k in plan_months},
    actual_by_month={int(k): int(v) for k, v in d26.groupby("m")["n_tests"].sum().items()},
    actual25_by_month={int(k): int(v) for k, v in d25.groupby("m")["n_tests"].sum().items()},
)
short_tbl = pd.DataFrame(cities).T[["t25", "t26", "plan", "growth", "plan_growth", "short"]]
display(short_tbl.style.format({"t25": "{:,.0f}", "t26": "{:,.0f}", "plan": "{:,.0f}", "short": "{:,.0f}", "growth": "{:+.1%}", "plan_growth": "{:+.1%}"}))
print("Total shortfall:", f"{deck['plan']['total_plan'] - deck['plan']['total26']:,}")


In [ ]:
# B: Chennai and Jaipur, quarter by quarter (Q1 = legacy in both years, Q2 = Nova in 2026)
mig_ = dm[dm["city"].isin(["Chennai", "Jaipur"]) & (dm["m"] <= 6) & dm["year"].isin([2025, 2026])].copy()
mig_["q"] = np.where(mig_["m"] <= 3, "Q1", "Q2")
two = {}
for c in ["Chennai", "Jaipur"]:
    for q in ["Q1", "Q2"]:
        row = {}
        for y in [2025, 2026]:
            d = mig_[(mig_["city"] == c) & (mig_["q"] == q) & (mig_["year"] == y)]
            row[y] = dict(
                visits=len(d), patients=int(d["person_id"].nunique()), tests=int(d["n_tests"].sum()),
                collected=int(d["collected_tests"].sum()), results=int(d["result_tests"].sum()),
                no_show=float(d["status"].eq("no_show").mean()), cancel=float(d["status"].eq("cancelled_or_missing").mean()),
                tpv=float(d["n_tests"].sum() / len(d)), res_per_booked=float(d["result_tests"].sum() / d["n_tests"].sum()),
            )
        two[f"{c}_{q}"] = row
deck["two"] = two

deck["two_monthly"] = {
    c: {f"{y}-{m:02d}": int(((dm["city"] == c) & (dm["year"] == y) & (dm["m"] == m)).sum()) for y in [2025, 2026] for m in range(1, 7)}
    for c in ["Chennai", "Jaipur"]
}
deck["visits_q"] = {}
for c in sorted(dm["city"].unique()):
    for q, (lo, hi) in {"Q1": (1, 3), "Q2": (4, 6)}.items():
        v25 = int(((dm["city"] == c) & (dm["year"] == 2025) & dm["m"].between(lo, hi)).sum())
        v26 = int(((dm["city"] == c) & (dm["year"] == 2026) & dm["m"].between(lo, hi)).sum())
        deck["visits_q"][f"{c}_{q}"] = dict(v25=v25, v26=v26, g=v26 / v25 - 1)

# Counting vs real: same three measures, both systems
view = pd.DataFrame({k: {m_: v[2026][m_] / v[2025][m_] - 1 for m_ in ["visits", "tests", "collected", "results"]} for k, v in two.items()}).T
display(view.style.format("{:+.1%}"))
print("Share of bookings with no collection record (cancelled_or_missing):")
print({k: f"{v[2025]['cancel']:.1%} -> {v[2026]['cancel']:.1%}" for k, v in two.items()})


In [ ]:
# EXTRA (not in Brief A or B): revenue and package mix. Revenue = test list price + home fee at booking, not invoiced revenue.
rev = {}
for nm, d in [("y25", d25), ("y26", d26)]:
    rev[nm] = dict(
        revenue=int(d["amount_rs"].sum()), tests=int(d["n_tests"].sum()), bookings=len(d),
        rev_per_test=float(d["amount_rs"].sum() / d["n_tests"].sum()), rev_per_visit=float(d["amount_rs"].mean()),
        pkg_share=float(d["has_package"].mean()),
    )
deck["rev"] = rev
print(f"Revenue Rs {rev['y25']['revenue']:,} -> Rs {rev['y26']['revenue']:,} ({rev['y26']['revenue'] / rev['y25']['revenue'] - 1:+.1%})")
print(f"Revenue per test {rev['y25']['rev_per_test']:.0f} -> {rev['y26']['rev_per_test']:.0f}")
ch_ = pd.DataFrame({k: v["g"] for k, v in deck["channel"].items()}).T
display(ch_.style.format("{:+.1%}"))


In [ ]:
# Cleaning audit trail (uses counters from section 2) and export
deck["clean"] = dict(
    n_raw_legacy=int(n_raw_legacy), n_raw_nova=int(n_raw_nova), n_raw_rows=int(n_raw_rows),
    n_header_rows=int(n_header_rows), n_bang=int(n_bangalore),
    n_notests=int((rejects_df["reason"] == "no tests").sum()),
    n_badslot_rows=int((rejects_df["reason"] == "slot_at before created_at").sum()),
    n_badslot_bk=int(rejects_df.loc[rejects_df["reason"] == "slot_at before created_at", "booking_id"].nunique()),
    n_nullcode=int(bookings_all["test_code"].isna().sum()),
    n_id_flag_bookings=int(bookings_all.loc[bookings_all["id_flag"], "booking_id"].nunique()),
    n_bookings_pre=len(bookings), n_tests_pre=int(bookings["n_tests"].sum()),
    n_superseded=int(is_superseded.sum()), n_current=len(current_bookings), n_current_tests=int(current_bookings["n_tests"].sum()),
    n_dups=int(dups.sum()), n_coll_kept=len(coll_clean), n_skew=int(skew.sum()),
    n_person_changed=int((bookings_final["patient_id"] != bookings_final["person_id"]).sum()),
    n_persons=int(bookings_final["person_id"].nunique()),
    status={k: int(v) for k, v in bookings_final["status"].value_counts().items()},
)
deck["resched"] = dict(replacements=int(current_bookings["rescheduled_from"].notna().sum()))

with open("deck_metrics.json", "w") as fh:
    json.dump(deck, fh, indent=1, default=float)
print("Wrote deck_metrics.json")


## 3b. Effect of each rule on the headline, and the counting split for Brief B

Two things the slide template asks for:
- **Appendix A** ("the decisions that moved the headline"): for each rule, the rows it touched and what H1 test growth would be without it.
- **Brief B**: how much of the apparent fall in Chennai and Jaipur is counting. Legacy exports one row per test line; Nova exports one object per booking with the tests nested inside. Counting rows in both would make Nova look much smaller.

`DECK-METRICS-3B` marker: do not delete, it stops this section being added twice.

In [ ]:
# DECK-METRICS-3B: effect of each rule on H1 test growth (Appendix A)
def h1_sum(df, col="n_tests", date="slot_at"):
    y, m = df[date].dt.year, df[date].dt.month
    return {yr: float(df.loc[(y == yr) & (m <= 6), col].sum()) for yr in (2025, 2026)}

base = h1_sum(current_bookings)
g_base = base[2026] / base[2025] - 1

# Rule: drop the old version of a rescheduled Nova booking
sup_bk = bookings.loc[is_superseded]
add_sup = h1_sum(sup_bk)
g_keep_superseded = (base[2026] + add_sup[2026]) / (base[2025] + add_sup[2025]) - 1

# Rule: reject bookings whose slot is before their creation time
bs = rejects_df.loc[rejects_df["reason"] == "slot_at before created_at"].copy()
bs["_t"] = bs["test_code"].notna().astype(int)
add_bs = h1_sum(bs, col="_t")
g_keep_badslot = (base[2026] + add_bs[2026]) / (base[2025] + add_bs[2025]) - 1

# Rule: merge duplicate patient IDs into one person
pat_raw = {yr: int(dm.loc[(dm["year"] == yr) & (dm["m"] <= 6), "patient_id"].nunique()) for yr in (2025, 2026)}
pat_person = {yr: int(dm.loc[(dm["year"] == yr) & (dm["m"] <= 6), "person_id"].nunique()) for yr in (2025, 2026)}

deck["rules"] = dict(
    base=base, g_base=g_base,
    superseded=dict(bookings=int(is_superseded.sum()), tests_h1_2026=add_sup[2026], g_without_rule=g_keep_superseded),
    bad_slot=dict(bookings=int(bs["booking_id"].nunique()), tests_h1_2025=add_bs[2025], tests_h1_2026=add_bs[2026], g_without_rule=g_keep_badslot),
    patients=dict(raw=pat_raw, person=pat_person,
                  g_raw=pat_raw[2026] / pat_raw[2025] - 1, g_person=pat_person[2026] / pat_person[2025] - 1),
)
r = deck["rules"]
print(f"Headline {g_base:+.3%} | keep superseded {g_keep_superseded:+.3%} | keep bad-slot {g_keep_badslot:+.3%}")
print(f"Patients growth: raw IDs {r['patients']['g_raw']:+.2%} vs merged {r['patients']['g_person']:+.2%}")


In [ ]:
# Brief B: counting split for Chennai and Jaipur, April to June
def q2(df, year, col="slot_at"):
    return df[(df[col].dt.year == year) & df[col].dt.month.between(4, 6)]

counting = {}
for c in ["Chennai", "Jaipur"]:
    leg25 = q2(legacy[legacy["city"].astype(str).str.strip().str.lower() == c.lower()], 2025)
    nov26 = q2(nova[nova["city"].astype(str).str.strip().str.lower() == c.lower()], 2026)
    sup_c = sup_bk[(sup_bk["city"] == c)]
    counting[c] = dict(
        legacy_rows_q2_2025=int(len(leg25)),                       # one row per test line
        nova_objects_q2_2026=int(nov26["booking_id"].nunique()),   # one object per booking
        nova_rows_q2_2026=int(len(nov26)),                         # after flattening: one row per test line
        superseded_q2_2026=int(len(q2(sup_c, 2026))),
        visits_q2_2025=deck["two"][f"{c}_Q2"][2025]["visits"], visits_q2_2026=deck["two"][f"{c}_Q2"][2026]["visits"],
    )
deck["counting"] = counting
tbl = pd.DataFrame(counting).T
tbl["naive_change"] = tbl["nova_objects_q2_2026"] / tbl["legacy_rows_q2_2025"] - 1
tbl["flattened_change_rows"] = tbl["nova_rows_q2_2026"] / tbl["legacy_rows_q2_2025"] - 1
tbl["real_visits_change"] = tbl["visits_q2_2026"] / tbl["visits_q2_2025"] - 1
display(tbl.style.format({"naive_change": "{:+.1%}", "flattened_change_rows": "{:+.1%}", "real_visits_change": "{:+.1%}"}))

with open("deck_metrics.json", "w") as fh:
    json.dump(deck, fh, indent=1, default=float)
print("Updated deck_metrics.json")
